# CLV Frequency Modeling

## Overview
This notebook trains and evaluates four predictive models for customer purchase frequency:
* **Linear Regression (Drivers-Only)**: Uses 9 behavioral engagement features
* **GBT (Drivers-Only)**: Gradient Boosted Trees with same 9 features
* **Linear Regression (Enhanced)**: Adds log-transformed baseline controls (past frequency, recency, tenure)
* **GBT (Enhanced)**: Gradient Boosted Trees with full feature set

## 1. Setup & Configuration

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import DoubleType
import numpy as np
import pandas as pd

# Configuration
TRAIN_TABLE = "dev_datalake_insight_analytics_spain.member_snapshot_training_es_v5"
OUTPUT_SCHEMA = "dev_datalake_insight_analytics_spain"

# Feature set A: Drivers-only (9 engagement features)
FEATURE_DRIVERS = [
    "is_omnichannel",
    "is_contactable",
    "is_ump_member",
    "is_multi_category_9p_universes",
    "is_fast_second_purchase_60d",
    "is_recent_buyer_60d",
    "has_ecosystem_engagement",
    "is_app_user",
    "is_low_returns"
]

# Feature set B: Enhanced (drivers + log-transformed baseline controls)
FEATURE_ENHANCED = FEATURE_DRIVERS + [
    "freq_12m_log",
    "recency_log",
    "tenure_log",
    "has_history_before_snapshot"
]

TARGET_COL = "future_freq_12m"

print(f"Training table: {TRAIN_TABLE}")
print(f"Feature sets: {len(FEATURE_DRIVERS)} drivers, {len(FEATURE_ENHANCED)} enhanced")
print(f"Target: {TARGET_COL}")

✓ Configuration loaded
✓ Training table: dev_datalake_insight_analytics_spain.member_snapshot_training_es_v5

📊 Feature Sets:
  Drivers-only: 9 features
    - is_omnichannel
    - is_contactable
    - is_ump_member
    - is_multi_category_9p_universes
    - is_fast_second_purchase_60d
    - is_recent_buyer_60d
    - has_ecosystem_engagement
    - is_app_user
    - is_low_returns
  Enhanced: 13 features (drivers + log-transformed baseline controls)
  Note: Using log1p transforms to prevent overflow

✓ Target: future_freq_12m


## 2. Data Loading & Preparation

In [0]:
# Load training data
df_raw = spark.table(TRAIN_TABLE)

# Sampling configuration
USE_SAMPLE = True
SAMPLE_FRACTION = 0.02  # 2% sample for faster GBT training on 2xlarge cluster
SAMPLE_SEED = 42

if USE_SAMPLE:
    print(f"Using {SAMPLE_FRACTION*100:.0f}% sample for development")
    df_raw = df_raw.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SAMPLE_SEED)
    df_raw.cache()
else:
    print("Using full dataset")

total_rows = df_raw.count()
print(f"Total rows: {total_rows:,}")

# Time-based split
df_train_full = df_raw.filter(F.col("snapshot_date") <= "2024-06-30")
df_validate = df_raw.filter(
    (F.col("snapshot_date") >= "2024-07-31") & 
    (F.col("snapshot_date") <= "2024-12-31")
)

# Tuning split within training data
df_train_tuning = df_train_full.filter(F.col("snapshot_date") <= "2024-03-31")
df_val_tuning = df_train_full.filter(
    (F.col("snapshot_date") >= "2024-04-30") & 
    (F.col("snapshot_date") <= "2024-06-30")
)

train_count = df_train_full.count()
val_count = df_validate.count()
train_tuning_count = df_train_tuning.count()
val_tuning_count = df_val_tuning.count()

print(f"\nFull train (≤2024-06-30): {train_count:,}")
print(f"  Tuning train (≤2024-03-31): {train_tuning_count:,}")
print(f"  Tuning val (2024-04-30 to 2024-06-30): {val_tuning_count:,}")
print(f"Validation (2024-07-31 to 2024-12-31): {val_count:,}")

# Store original size for reference
total_rows_original = 261_171_384

Loading training data from v5...
Sampling 2% of data for development/tuning...
Total rows (after sampling): 5,223,647

Full train set (≤ 2024-06-30): 3,918,201 rows
  Tuning train (≤ 2024-03-31): 3,263,879 rows
  Tuning val (2024-04-30 to 2024-06-30): 654,322 rows
Final validation set (2024-07-31 to 2024-12-31): 1,305,446 rows
Train/Val split: 75.0% / 25.0%


In [0]:
# Create log-transformed features and label
for df_name, df_obj in [("train_full", df_train_full), ("train_tuning", df_train_tuning),
                         ("val_tuning", df_val_tuning), ("validate", df_validate)]:
    
    # Log-transformed baseline features
    df_obj = df_obj.withColumn(
        "freq_12m_log",
        F.log1p(F.col("freq_12m").cast(DoubleType()))
    ).withColumn(
        "recency_log",
        F.log1p(F.coalesce(F.col("recency_days"), F.lit(0)).cast(DoubleType()))
    ).withColumn(
        "tenure_log",
        F.log1p(F.coalesce(F.col("customer_tenure_months"), F.lit(0.0)))
    )
    
    # Log-transformed target
    df_obj = df_obj.withColumn(
        "label",
        F.log1p(F.col(TARGET_COL).cast(DoubleType()))
    )
    
    # Store back
    if df_name == "train_full":
        df_train_full = df_obj
    elif df_name == "train_tuning":
        df_train_tuning = df_obj
    elif df_name == "val_tuning":
        df_val_tuning = df_obj
    else:
        df_validate = df_obj

print("Features created: freq_12m_log, recency_log, tenure_log, label")

# Create feature vectors
assembler_drivers = VectorAssembler(
    inputCols=FEATURE_DRIVERS,
    outputCol="features_drivers",
    handleInvalid="skip"
)

assembler_enhanced = VectorAssembler(
    inputCols=FEATURE_ENHANCED,
    outputCol="features_enhanced",
    handleInvalid="skip"
)

# Apply assemblers
for df_name, df_obj in [("train_full", df_train_full), ("train_tuning", df_train_tuning),
                         ("val_tuning", df_val_tuning), ("validate", df_validate)]:
    df_obj = assembler_drivers.transform(df_obj)
    df_obj = assembler_enhanced.transform(df_obj)
    
    if df_name == "train_full":
        df_train_full = df_obj
    elif df_name == "train_tuning":
        df_train_tuning = df_obj
    elif df_name == "val_tuning":
        df_val_tuning = df_obj
    else:
        df_validate = df_obj

print(f"Feature vectors: features_drivers ({len(FEATURE_DRIVERS)}), features_enhanced ({len(FEATURE_ENHANCED)})")

# Cache for training
df_train_tuning.cache()
df_val_tuning.cache()
df_train_full.cache()
df_validate.cache()

# Target distribution
print("\nTarget distribution:")
df_train_full.select(
    F.min(TARGET_COL).alias("min"),
    F.expr("percentile_approx(future_freq_12m, 0.50)").alias("median"),
    F.max(TARGET_COL).alias("max"),
    F.mean(TARGET_COL).alias("mean")
).show()

Preparing features and target...
✓ Log-transformed baseline features created (freq_12m_log, recency_log, tenure_log)
✓ Target transformed: label = log1p(future_freq_12m)

Creating feature vectors...
✓ features_drivers assembled (9 features)
✓ features_enhanced assembled (13 features with log-transforms)

Target distribution (original scale):
+---+---+------+---+----+-----------------+
|min|p25|median|p75| max|             mean|
+---+---+------+---+----+-----------------+
|  0|  0|     1|  4|1068|2.974324951680631|
+---+---+------+---+----+-----------------+



## 3. Model Training - Drivers Only (9 Features)

In [0]:
# Train Linear Regression on drivers-only features with hyperparameter tuning
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

param_grid = [
    {"regParam": 0.0, "elasticNetParam": 0.0},
    {"regParam": 0.0, "elasticNetParam": 0.5},
    {"regParam": 0.0, "elasticNetParam": 1.0},
    {"regParam": 0.01, "elasticNetParam": 0.0},
    {"regParam": 0.01, "elasticNetParam": 0.5},
    {"regParam": 0.01, "elasticNetParam": 1.0},
    {"regParam": 0.1, "elasticNetParam": 0.0},
    {"regParam": 0.1, "elasticNetParam": 0.5},
    {"regParam": 0.1, "elasticNetParam": 1.0},
]

print(f"Tuning {len(param_grid)} parameter combinations...")

best_rmse = float('inf')
best_params_lr_drivers = None

for params in param_grid:
    lr = LinearRegression(
        featuresCol="features_drivers",
        labelCol="label",
        predictionCol="prediction",
        maxIter=100,
        standardization=True,
        regParam=params["regParam"],
        elasticNetParam=params["elasticNetParam"]
    )
    
    model = lr.fit(df_train_tuning)
    predictions = model.transform(df_val_tuning)
    rmse = evaluator.evaluate(predictions)
    
    if rmse < best_rmse:
        best_rmse = rmse
        best_params_lr_drivers = params

print(f"Best params: regParam={best_params_lr_drivers['regParam']}, elasticNetParam={best_params_lr_drivers['elasticNetParam']}")
print(f"Tuning validation RMSE: {best_rmse:.4f}")

# Retrain on full training set
print(f"\nRetraining on full train set...")
lr_drivers_final = LinearRegression(
    featuresCol="features_drivers",
    labelCol="label",
    predictionCol="prediction",
    maxIter=100,
    standardization=True,
    regParam=best_params_lr_drivers["regParam"],
    elasticNetParam=best_params_lr_drivers["elasticNetParam"]
)

model_lr_drivers = lr_drivers_final.fit(df_train_full)

# Extract coefficients
coefficients = model_lr_drivers.coefficients.toArray()
intercept = model_lr_drivers.intercept

coef_data_drivers = []
for i, (feature, coef) in enumerate(zip(FEATURE_DRIVERS, coefficients)):
    coef_data_drivers.append({
        "driver": feature,
        "coefficient": float(coef),
        "abs_coefficient": abs(float(coef)),
        "direction": "positive" if coef > 0 else "negative" if coef < 0 else "zero",
        "feature_set": "drivers_only"
    })

coef_data_drivers.append({
    "driver": "intercept",
    "coefficient": float(intercept),
    "abs_coefficient": abs(float(intercept)),
    "direction": "positive" if intercept > 0 else "negative",
    "feature_set": "drivers_only"
})

driver_weights_lr_drivers = spark.createDataFrame(coef_data_drivers)
driver_weights_lr_drivers = driver_weights_lr_drivers.orderBy(F.desc("abs_coefficient"))

print(f"\nLR (Drivers) trained. Intercept: {intercept:.4f}")
print(f"Coefficients computed for {len(FEATURE_DRIVERS)} features")

MODEL 1/4: Linear Regression (Drivers-Only)
Using time-based tuning split to avoid time mixing

Testing 9 parameter combinations...
  regParam=0.0, elasticNetParam=0.0: RMSE=0.7559
  regParam=0.0, elasticNetParam=0.5: RMSE=0.7559
  regParam=0.0, elasticNetParam=1.0: RMSE=0.7559
  regParam=0.01, elasticNetParam=0.0: RMSE=0.7557
  regParam=0.01, elasticNetParam=0.5: RMSE=0.7554
  regParam=0.01, elasticNetParam=1.0: RMSE=0.7553
  regParam=0.1, elasticNetParam=0.0: RMSE=0.7551
  regParam=0.1, elasticNetParam=0.5: RMSE=0.7609
  regParam=0.1, elasticNetParam=1.0: RMSE=0.7757

✓ Best parameters found:
  regParam: 0.1
  elasticNetParam: 0.0
  Tuning validation RMSE: 0.7551

Retraining on full train set with best params...
✓ LR (Drivers-Only) trained on full train set
  Intercept: 0.8267
  Num features: 9



In [0]:
# Train GBT on drivers-only features (using sensible defaults)
print("Training GBT with default parameters (maxDepth=5, maxIter=50, stepSize=0.1)...\n")

gbt_drivers_final = GBTRegressor(
    featuresCol="features_drivers",
    labelCol="label",
    predictionCol="prediction",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

model_gbt_drivers = gbt_drivers_final.fit(df_train_full)

print(f"GBT (Drivers) trained: {model_gbt_drivers.getNumTrees} trees, {model_gbt_drivers.totalNumNodes} total nodes")

# Extract feature importances
importances = model_gbt_drivers.featureImportances.toArray()

imp_data_drivers = []
for i, (feature, importance) in enumerate(zip(FEATURE_DRIVERS, importances)):
    imp_data_drivers.append({
        "driver": feature,
        "importance": float(importance),
        "feature_set": "drivers_only"
    })

driver_importance_gbt_drivers = spark.createDataFrame(imp_data_drivers)
driver_importance_gbt_drivers = driver_importance_gbt_drivers.orderBy(F.desc("importance"))

window = Window.orderBy(F.desc("importance"))
driver_importance_gbt_drivers = driver_importance_gbt_drivers.withColumn("rank", F.row_number().over(window))

print("\nFeature importances:")
driver_importance_gbt_drivers.show(10, truncate=False)

best_params_gbt_drivers = {"maxDepth": 5, "maxIter": 50, "stepSize": 0.1}

MODEL 2/4: Gradient Boosted Trees (Drivers-Only)
Training with sensible default parameters (skipping tuning for speed)

Using default configuration:
  maxDepth: 5
  maxIter: 50
  stepSize: 0.1
  subsamplingRate: 0.8
  seed: 42

Training GBT on full train set...

✓ GBT (Drivers-Only) trained on full train set
  Number of trees: 50
  Max depth: 5
  Total nodes: 3150

✓ Training complete



## 4. Model Training - Enhanced (13 Features)
Adds log-transformed baseline controls: past frequency, recency, tenure

In [0]:
# Train Linear Regression on enhanced features with hyperparameter tuning
print(f"Tuning {len(param_grid)} parameter combinations...")

best_rmse_lr_enh = float('inf')
best_params_lr_enhanced = None

for params in param_grid:
    lr = LinearRegression(
        featuresCol="features_enhanced",
        labelCol="label",
        predictionCol="prediction",
        maxIter=100,
        standardization=True,
        regParam=params["regParam"],
        elasticNetParam=params["elasticNetParam"]
    )
    
    model = lr.fit(df_train_tuning)
    predictions = model.transform(df_val_tuning)
    rmse = evaluator.evaluate(predictions)
    
    if rmse < best_rmse_lr_enh:
        best_rmse_lr_enh = rmse
        best_params_lr_enhanced = params

print(f"Best params: regParam={best_params_lr_enhanced['regParam']}, elasticNetParam={best_params_lr_enhanced['elasticNetParam']}")
print(f"Tuning validation RMSE: {best_rmse_lr_enh:.4f}")

# Retrain on full training set
print(f"\nRetraining on full train set...")
lr_enhanced_final = LinearRegression(
    featuresCol="features_enhanced",
    labelCol="label",
    predictionCol="prediction",
    maxIter=100,
    standardization=True,
    regParam=best_params_lr_enhanced["regParam"],
    elasticNetParam=best_params_lr_enhanced["elasticNetParam"]
)

model_lr_enhanced = lr_enhanced_final.fit(df_train_full)

# Extract coefficients
coefficients_enh = model_lr_enhanced.coefficients.toArray()
intercept_enh = model_lr_enhanced.intercept

coef_data_enhanced = []
for i, (feature, coef) in enumerate(zip(FEATURE_ENHANCED, coefficients_enh)):
    coef_data_enhanced.append({
        "driver": feature,
        "coefficient": float(coef),
        "abs_coefficient": abs(float(coef)),
        "direction": "positive" if coef > 0 else "negative" if coef < 0 else "zero",
        "feature_set": "enhanced"
    })

coef_data_enhanced.append({
    "driver": "intercept",
    "coefficient": float(intercept_enh),
    "abs_coefficient": abs(float(intercept_enh)),
    "direction": "positive" if intercept_enh > 0 else "negative",
    "feature_set": "enhanced"
})

driver_weights_lr_enhanced = spark.createDataFrame(coef_data_enhanced)
driver_weights_lr_enhanced = driver_weights_lr_enhanced.orderBy(F.desc("abs_coefficient"))

print(f"\nLR (Enhanced) trained. Intercept: {intercept_enh:.4f}")
print(f"Coefficients computed for {len(FEATURE_ENHANCED)} features")

MODEL 3/4: Linear Regression (Enhanced)
Using best params from drivers-only tuning + retraining on enhanced features

Testing 9 parameter combinations on enhanced features...
  regParam=0.0, elasticNetParam=0.0: RMSE=0.7180
  regParam=0.0, elasticNetParam=0.5: RMSE=0.7180
  regParam=0.0, elasticNetParam=1.0: RMSE=0.7180
  regParam=0.01, elasticNetParam=0.0: RMSE=0.7152
  regParam=0.01, elasticNetParam=0.5: RMSE=0.7140
  regParam=0.01, elasticNetParam=1.0: RMSE=0.7142
  regParam=0.1, elasticNetParam=0.0: RMSE=0.7206
  regParam=0.1, elasticNetParam=0.5: RMSE=0.7386
  regParam=0.1, elasticNetParam=1.0: RMSE=0.7533

✓ Best parameters found:
  regParam: 0.01
  elasticNetParam: 0.5
  Tuning validation RMSE: 0.7140

Retraining on full train set...
✓ LR (Enhanced) trained on full train set
  Intercept: 0.6226
  Num features: 13



In [0]:
# Train GBT on enhanced features (using sensible defaults)
print("Training GBT with default parameters (maxDepth=5, maxIter=50, stepSize=0.1)...\n")

gbt_enhanced_final = GBTRegressor(
    featuresCol="features_enhanced",
    labelCol="label",
    predictionCol="prediction",
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

model_gbt_enhanced = gbt_enhanced_final.fit(df_train_full)

print(f"GBT (Enhanced) trained: {model_gbt_enhanced.getNumTrees} trees, {model_gbt_enhanced.totalNumNodes} total nodes")

# Extract feature importances
importances_enh = model_gbt_enhanced.featureImportances.toArray()

imp_data_enhanced = []
for i, (feature, importance) in enumerate(zip(FEATURE_ENHANCED, importances_enh)):
    imp_data_enhanced.append({
        "driver": feature,
        "importance": float(importance),
        "feature_set": "enhanced"
    })

driver_importance_gbt_enhanced = spark.createDataFrame(imp_data_enhanced)
driver_importance_gbt_enhanced = driver_importance_gbt_enhanced.orderBy(F.desc("importance"))

window_enh = Window.orderBy(F.desc("importance"))
driver_importance_gbt_enhanced = driver_importance_gbt_enhanced.withColumn("rank", F.row_number().over(window_enh))

print("\nFeature importances:")
driver_importance_gbt_enhanced.show(13, truncate=False)

best_params_gbt_enhanced = {"maxDepth": 5, "maxIter": 50, "stepSize": 0.1}

MODEL 4/4: Gradient Boosted Trees (Enhanced)
Training with sensible default parameters (skipping tuning for speed)

Using default configuration:
  maxDepth: 5
  maxIter: 50
  stepSize: 0.1
  subsamplingRate: 0.8
  seed: 42

Training GBT on full train set...

✓ GBT (Enhanced) trained on full train set
  Number of trees: 50
  Max depth: 5
  Total nodes: 3148

✓ Training complete



## 5. Model Evaluation

In [0]:
# Generate predictions from all 4 models
print("Generating predictions on validation set...\n")

# Model 1: LR (Drivers)
df_val_lr_drivers = model_lr_drivers.transform(df_validate)
df_val_lr_drivers = df_val_lr_drivers.withColumn(
    "y_pred_lr_drivers",
    F.greatest(F.lit(0.0), F.expm1(F.col("prediction")))
).select("member_id", "snapshot_date", F.col(TARGET_COL).alias("y_true"), "y_pred_lr_drivers")

# Model 2: GBT (Drivers)
df_val_gbt_drivers = model_gbt_drivers.transform(df_validate)
df_val_gbt_drivers = df_val_gbt_drivers.withColumn(
    "y_pred_gbt_drivers",
    F.greatest(F.lit(0.0), F.expm1(F.col("prediction")))
).select("member_id", "snapshot_date", "y_pred_gbt_drivers")

# Model 3: LR (Enhanced) - clip in log space to prevent overflow
df_val_lr_enhanced = model_lr_enhanced.transform(df_validate)
df_val_lr_enhanced = df_val_lr_enhanced.withColumn(
    "prediction_clipped",
    F.least(F.greatest(F.col("prediction"), F.lit(0.0)), F.lit(10.0))
).withColumn(
    "y_pred_lr_enhanced",
    F.greatest(F.lit(0.0), F.expm1(F.col("prediction_clipped")))
).select("member_id", "snapshot_date", "y_pred_lr_enhanced")

# Model 4: GBT (Enhanced) - clip in log space for consistency
df_val_gbt_enhanced = model_gbt_enhanced.transform(df_validate)
df_val_gbt_enhanced = df_val_gbt_enhanced.withColumn(
    "prediction_clipped",
    F.least(F.greatest(F.col("prediction"), F.lit(0.0)), F.lit(10.0))
).withColumn(
    "y_pred_gbt_enhanced",
    F.greatest(F.lit(0.0), F.expm1(F.col("prediction_clipped")))
).select("member_id", "snapshot_date", "y_pred_gbt_enhanced")

# Combine all predictions
df_predictions = df_val_lr_drivers \
    .join(df_val_gbt_drivers, on=["member_id", "snapshot_date"], how="inner") \
    .join(df_val_lr_enhanced, on=["member_id", "snapshot_date"], how="inner") \
    .join(df_val_gbt_enhanced, on=["member_id", "snapshot_date"], how="inner")

df_predictions.cache()
pred_count = df_predictions.count()

print(f"Predictions generated for {pred_count:,} validation records")

Generating predictions on validation set from all 4 models...

1/4: LR (Drivers-Only)...
2/4: GBT (Drivers-Only)...
3/4: LR (Enhanced) with prediction clipping...
4/4: GBT (Enhanced) with prediction clipping...

✓ Predictions generated for 1,305,446 validation records
✓ Drivers-only: predictions clamped to ≥ 0
✓ Enhanced: predictions clipped in log space [0, 10] then transformed

Sample predictions:
+------------------------------------+-------------+------+------------------+------------------+-------------------+-------------------+
|member_id                           |snapshot_date|y_true|y_pred_lr_drivers |y_pred_gbt_drivers|y_pred_lr_enhanced |y_pred_gbt_enhanced|
+------------------------------------+-------------+------+------------------+------------------+-------------------+-------------------+
|50878f48-ed4e-404b-9604-4f73ca82c3ef|2024-12-31   |2     |2.9242416368539446|3.991385751453544 |5.22521402244187   |6.346183730839226  |
|35f2d2a3-628e-4c8b-83de-5ed15f3538af|2024-09

In [0]:
# Evaluate all 4 models on validation set
print("Evaluating all 4 models on validation set...\n")

# Helper functions
def compute_metrics(df, pred_col):
    mae = df.select(F.mean(F.abs(F.col("y_true") - F.col(pred_col)))).collect()[0][0]
    rmse = df.select(F.sqrt(F.mean(F.pow(F.col("y_true") - F.col(pred_col), 2)))).collect()[0][0]
    return mae, rmse

def compute_spearman(df, pred_col):
    df_ranks = df.select(
        F.dense_rank().over(Window.orderBy("y_true")).alias("rank_true"),
        F.dense_rank().over(Window.orderBy(pred_col)).alias("rank_pred")
    )
    return df_ranks.stat.corr("rank_true", "rank_pred")

def compute_lift(df, pred_col, overall_avg):
    df_decile = df.withColumn("decile", F.ntile(10).over(Window.orderBy(F.desc(pred_col))))
    top10_actual = df_decile.filter(F.col("decile") == 1).agg(F.mean("y_true")).collect()[0][0]
    return top10_actual / overall_avg

# Compute overall average
overall_actual = df_predictions.agg(F.mean("y_true")).collect()[0][0]
print(f"Overall average frequency: {overall_actual:.2f}\n")

# Model definitions
models = [
    ("LR_drivers", "drivers_only", "y_pred_lr_drivers"),
    ("GBT_drivers", "drivers_only", "y_pred_gbt_drivers"),
    ("LR_enhanced", "enhanced", "y_pred_lr_enhanced"),
    ("GBT_enhanced", "enhanced", "y_pred_gbt_enhanced")
]

metrics_dict = {}

print("Model Performance:")
print("-" * 80)

for model_name, feature_set, pred_col in models:
    mae, rmse = compute_metrics(df_predictions, pred_col)
    spearman = compute_spearman(df_predictions, pred_col)
    lift = compute_lift(df_predictions, pred_col, overall_actual)
    
    metrics_dict[model_name] = {
        "feature_set": feature_set,
        "MAE": mae,
        "RMSE": rmse,
        "Spearman": float(spearman),
        "decile_lift_top10": float(lift)
    }
    
    print(f"{model_name:20s} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | Spearman: {spearman:.4f} | Lift: {lift:.2f}x")

print("-" * 80)

Evaluating all 4 models on validation set (original scale)...

LR_drivers           | Feature Set: drivers_only    | MAE: 2.3350 | RMSE: 3.9419
GBT_drivers          | Feature Set: drivers_only    | MAE: 2.3201 | RMSE: 3.8792
LR_enhanced          | Feature Set: enhanced        | MAE: 2.1458 | RMSE: 3.6178
GBT_enhanced         | Feature Set: enhanced        | MAE: 2.1134 | RMSE: 3.5498

✓ MAE and RMSE computed for all 4 models


## 6. Save Results to Delta Tables

In [0]:
# Save predictions to Delta table
predictions_table = f"{OUTPUT_SCHEMA}.clv_freq_predictions_v5"

df_predictions.write.mode("overwrite").format("delta").saveAsTable(predictions_table)

print(f"Saved {pred_count:,} predictions to {predictions_table}")

Saving predictions to Delta table (v5)...
✓ Saved: dev_datalake_insight_analytics_spain.clv_freq_predictions_v5
  Rows: 1,305,446
  Columns: member_id, snapshot_date, y_true, y_pred_lr_drivers, y_pred_gbt_drivers, y_pred_lr_enhanced, y_pred_gbt_enhanced

Sample from saved table:
+------------------------------------+-------------+------+------------------+------------------+------------------+-------------------+
|member_id                           |snapshot_date|y_true|y_pred_lr_drivers |y_pred_gbt_drivers|y_pred_lr_enhanced|y_pred_gbt_enhanced|
+------------------------------------+-------------+------+------------------+------------------+------------------+-------------------+
|00087624-0fd3-4046-8a9f-33343248706f|2024-11-30   |5     |10.888364231371778|10.185101858955013|13.696336881543024|11.840564700316857 |
|002b03bc-1eee-4753-9cf8-a03a24becbe0|2024-11-30   |2     |1.368085762434407 |1.4593440309735388|1.3249633394904845|1.5212719723436607 |
|00383ce9-93e7-428c-a7f6-e061f41a40

In [0]:
# Save model metrics to Delta table
metrics_data = []
for model_name, metrics in metrics_dict.items():
    metrics_data.append({
        "model_name": model_name,
        "feature_set": metrics["feature_set"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "Spearman": metrics["Spearman"],
        "decile_lift_top10": metrics["decile_lift_top10"],
        "sample_fraction": SAMPLE_FRACTION if USE_SAMPLE else 1.0
    })

df_metrics = spark.createDataFrame(metrics_data)
metrics_table = f"{OUTPUT_SCHEMA}.clv_freq_metrics_v5"

df_metrics.write.mode("overwrite").format("delta").saveAsTable(metrics_table)

print(f"Saved metrics to {metrics_table}")
spark.table(metrics_table).orderBy("RMSE").show(truncate=False)

Saving metrics to Delta table (v5)...
✓ Saved: dev_datalake_insight_analytics_spain.clv_freq_metrics_v5

Final Model Metrics (All 4 Models):
+------------------+------------------+------------------+-----------------+------------+------------+---------------+
|MAE               |RMSE              |Spearman          |decile_lift_top10|feature_set |model_name  |sample_fraction|
+------------------+------------------+------------------+-----------------+------------+------------+---------------+
|2.1133737612633547|3.549765960865598 |0.6198907611437267|3.241548506687002|enhanced    |GBT_enhanced|0.02           |
|2.145754419640624 |3.617801266918702 |0.5814781458906141|3.204464569112438|enhanced    |LR_enhanced |0.02           |
|2.32011827561605  |3.8791712075186586|0.5581830887124191|3.068167225535854|drivers_only|GBT_drivers |0.02           |
|2.3350345329510063|3.9418851291339783|0.5540711717230988|3.059711877282108|drivers_only|LR_drivers  |0.02           |
+------------------+------

In [0]:
# Save linear model coefficients
df_coef_combined = driver_weights_lr_drivers.union(driver_weights_lr_enhanced)
weights_table = f"{OUTPUT_SCHEMA}.driver_weights_linear_v5"

df_coef_combined.write.mode("overwrite").format("delta").saveAsTable(weights_table)

print(f"Saved coefficients to {weights_table}")

Saving linear model driver weights (drivers-only + enhanced) to v5...
✓ Saved: dev_datalake_insight_analytics_spain.driver_weights_linear_v5
  Feature sets: drivers_only + enhanced

Top 10 Drivers (Drivers-Only):
+-------------------+-------------------+---------+------------------------------+------------+
|abs_coefficient    |coefficient        |direction|driver                        |feature_set |
+-------------------+-------------------+---------+------------------------------+------------+
|0.8267461073528739 |0.8267461073528739 |positive |intercept                     |drivers_only|
|0.7977114057597795 |0.7977114057597795 |positive |is_multi_category_9p_universes|drivers_only|
|0.4235743775656428 |0.4235743775656428 |positive |is_recent_buyer_60d           |drivers_only|
|0.2781507996208118 |0.2781507996208118 |positive |is_ump_member                 |drivers_only|
|0.25452252162604966|0.25452252162604966|positive |is_fast_second_purchase_60d   |drivers_only|
|0.2252712935547675

In [0]:
# Save GBT feature importances
df_imp_combined = driver_importance_gbt_drivers.union(driver_importance_gbt_enhanced)
importance_table = f"{OUTPUT_SCHEMA}.driver_importance_gbt_v5"

df_imp_combined.write.mode("overwrite").format("delta").saveAsTable(importance_table)

print(f"Saved importances to {importance_table}")

Saving GBT feature importances (drivers-only + enhanced) to v5...
✓ Saved: dev_datalake_insight_analytics_spain.driver_importance_gbt_v5
  Feature sets: drivers_only + enhanced

Top 10 Features (Drivers-Only):
+------------------------------+------------+--------------------+----+
|driver                        |feature_set |importance          |rank|
+------------------------------+------------+--------------------+----+
|is_recent_buyer_60d           |drivers_only|0.5080913897735301  |1   |
|is_multi_category_9p_universes|drivers_only|0.16416848010939625 |2   |
|is_fast_second_purchase_60d   |drivers_only|0.11136365976248053 |3   |
|is_contactable                |drivers_only|0.0717157638744944  |4   |
|is_omnichannel                |drivers_only|0.0505641260639292  |5   |
|has_ecosystem_engagement      |drivers_only|0.03267676499405713 |6   |
|is_ump_member                 |drivers_only|0.025597992638105544|7   |
|is_app_user                   |drivers_only|0.019710437559510047|8   

In [0]:
# Model comparison summary
print("=" * 80)
print("CLV FREQUENCY MODELING - SUMMARY")
print("=" * 80)

print(f"\nData: {TRAIN_TABLE}")
if USE_SAMPLE:
    print(f"Sample: {SAMPLE_FRACTION*100:.0f}% ({total_rows:,} rows)")
else:
    print(f"Full dataset: {total_rows:,} rows")
print(f"Train: {train_count:,} | Validation: {val_count:,}")

print(f"\nFeature Sets:")
print(f"  Drivers-Only: {len(FEATURE_DRIVERS)} features")
print(f"  Enhanced: {len(FEATURE_ENHANCED)} features (adds baseline controls)")

print(f"\nModel Performance (Validation):")
print(f"{'Model':<20} {'Features':<15} {'MAE':<8} {'RMSE':<8} {'Spearman':<10} {'Lift'}")
print("-" * 80)
for model_name, metrics in sorted(metrics_dict.items(), key=lambda x: x[1]['RMSE']):
    print(f"{model_name:<20} {metrics['feature_set']:<15} {metrics['MAE']:<8.4f} {metrics['RMSE']:<8.4f} {metrics['Spearman']:<10.4f} {metrics['decile_lift_top10']:.2f}x")

best_model = min(metrics_dict.items(), key=lambda x: x[1]['RMSE'])
print(f"\nBest Model: {best_model[0]} (RMSE: {best_model[1]['RMSE']:.4f})")

print(f"\nEnhanced vs Drivers Improvement:")
for model_type in ['LR', 'GBT']:
    drivers_key = f"{model_type}_drivers"
    enhanced_key = f"{model_type}_enhanced"
    
    if drivers_key in metrics_dict and enhanced_key in metrics_dict:
        delta_rmse = metrics_dict[drivers_key]['RMSE'] - metrics_dict[enhanced_key]['RMSE']
        pct_improvement = (delta_rmse / metrics_dict[drivers_key]['RMSE']) * 100
        print(f"  {model_type}: RMSE improved by {pct_improvement:+.2f}%")

print(f"\nSaved Tables:")
print(f"  {OUTPUT_SCHEMA}.clv_freq_predictions_v5")
print(f"  {OUTPUT_SCHEMA}.clv_freq_metrics_v5")
print(f"  {OUTPUT_SCHEMA}.driver_weights_linear_v5")
print(f"  {OUTPUT_SCHEMA}.driver_importance_gbt_v5")

if USE_SAMPLE:
    print(f"\nNote: Results based on {SAMPLE_FRACTION*100:.0f}% sample. Set USE_SAMPLE=False for full data.")
print("=" * 80)

CLV FREQUENCY MODELING - FINAL SUMMARY (v5)

📊 DATA:
  Training table: dev_datalake_insight_analytics_spain.member_snapshot_training_es_v5
  Sampling: 2% sample (seed=42)
  Full dataset size: 261,171,384 rows
  Sample size: 5,223,647 rows
  Train rows: 3,918,201 (≤ 2024-06-30)
  Validation rows: 1,305,446 (2024-07-31 to 2024-12-31)

🔧 FEATURE SETS:
  Drivers-Only: 9 features
  Enhanced: 13 features (drivers + 4 baseline controls)

🤖 MODELS TRAINED: 4
  1. Linear Regression (Drivers-Only)
  2. GBT (Drivers-Only)
  3. Linear Regression (Enhanced)
  4. GBT (Enhanced)

📈 MODEL PERFORMANCE (Validation Set):
Model                     Feature Set     MAE      RMSE     Spearman   Lift 10%
--------------------------------------------------------------------------------
GBT_enhanced              enhanced        2.1134   3.5498   0.6199     3.24x
LR_enhanced               enhanced        2.1458   3.6178   0.5815     3.20x
GBT_drivers               drivers_only    2.3201   3.8792   0.5582     3.07

## 7. Additional Analysis & Export

In [0]:
# Export results to CSV for download
import os

export_base_path = "/FileStore/clv_exports_v5/"

print("Exporting results to CSV...\n")

# Export metrics
df_metrics_export = spark.table("dev_datalake_insight_analytics_spain.clv_freq_metrics_v5")
df_metrics_export.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{export_base_path}clv_freq_metrics_v5")
print(f"1. Metrics: {export_base_path}clv_freq_metrics_v5/")

# Export linear weights
df_weights_export = spark.table("dev_datalake_insight_analytics_spain.driver_weights_linear_v5")
df_weights_export.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{export_base_path}driver_weights_linear_v5")
print(f"2. Linear Weights: {export_base_path}driver_weights_linear_v5/")

# Export GBT importances
df_importance_export = spark.table("dev_datalake_insight_analytics_spain.driver_importance_gbt_v5")
df_importance_export.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{export_base_path}driver_importance_gbt_v5")
print(f"3. GBT Importances: {export_base_path}driver_importance_gbt_v5/")

print(f"\nExport complete. Files available at: {export_base_path}")

Exporting v5 results to CSV...

1/3: Exporting clv_freq_metrics_v5...
   ✓ Exported to /FileStore/clv_exports_v5/clv_freq_metrics_v5/
2/3: Exporting driver_weights_linear_v5...
   ✓ Exported to /FileStore/clv_exports_v5/driver_weights_linear_v5/
3/3: Exporting driver_importance_gbt_v5...
   ✓ Exported to /FileStore/clv_exports_v5/driver_importance_gbt_v5/

✅ EXPORT COMPLETE

Files exported to: /FileStore/clv_exports_v5/

To download:
1. Go to the Databricks UI
2. Navigate to 'Data' > 'DBFS' > 'FileStore' > 'clv_exports_v5'
3. Click on each folder and download the .csv file

Or use these direct URLs (replace <databricks-instance> with your workspace URL):
   https://<databricks-instance>/files/clv_exports_v5/clv_freq_metrics_v5/?o=<workspace-id>
   https://<databricks-instance>/files/clv_exports_v5/driver_weights_linear_v5/?o=<workspace-id>
   https://<databricks-instance>/files/clv_exports_v5/driver_importance_gbt_v5/?o=<workspace-id>

Note: Each folder contains a single CSV file (part

In [0]:
# Compare feature importance: Drivers-Only vs Enhanced
df_importance = spark.table("dev_datalake_insight_analytics_spain.driver_importance_gbt_v5")

df_drivers_only = df_importance.filter(F.col("feature_set") == "drivers_only") \
    .select("driver", F.col("importance").alias("importance_drivers"))

df_enhanced_set = df_importance.filter(F.col("feature_set") == "enhanced") \
    .select("driver", F.col("importance").alias("importance_enhanced"))

df_comparison = df_drivers_only.join(df_enhanced_set, on="driver", how="inner") \
    .withColumn("pct_change", 
                ((F.col("importance_enhanced") - F.col("importance_drivers")) / 
                 F.col("importance_drivers") * 100)) \
    .orderBy(F.desc("importance_drivers"))

print("Feature Importance Change (Drivers-Only → Enhanced)\n")
print(f"{'Driver':<35} {'Drivers-Only':<15} {'Enhanced':<15} {'% Change'}")
print("-" * 80)

for row in df_comparison.collect():
    print(f"{row['driver']:<35} {row['importance_drivers']:<15.4f} {row['importance_enhanced']:<15.4f} {row['pct_change']:>8.1f}%")

FEATURE IMPORTANCE COMPARISON - DRIVERS-ONLY vs ENHANCED

📊 9 ENGAGEMENT DRIVERS - IMPORTANCE COMPARISON

How importance changes when we add historical features (past frequency, recency, tenure)

Driver                              Drivers-Only    Enhanced        Change          % Change
is_recent_buyer_60d                 0.5081 (100%)  0.0478        -0.4603     -90.6%     ⬇️⬇️⬇️ DROPS
is_multi_category_9p_universes      0.1642 (100%)  0.0036        -0.1606     -97.8%     ⬇️⬇️⬇️ DROPS
is_fast_second_purchase_60d         0.1114 (100%)  0.0239        -0.0874     -78.5%     ⬇️ drops
is_contactable                      0.0717 (100%)  0.0526        -0.0192     -26.7%     ⬇️ drops
is_omnichannel                      0.0506 (100%)  0.0285        -0.0220     -43.6%     ⬇️ drops
has_ecosystem_engagement            0.0327 (100%)  0.0172        -0.0155     -47.3%     ⬇️ drops
is_ump_member                       0.0256 (100%)  0.0126        -0.0130     -50.8%     ⬇️ drops
is_app_user             

In [0]:
# Precision@K: How well do we identify top buyers?
df_preds = spark.table("dev_datalake_insight_analytics_spain.clv_freq_predictions_v5")

# Define high-frequency threshold (top 20%)
p80_threshold = df_preds.approxQuantile("y_true", [0.80], 0.01)[0]
df_preds = df_preds.withColumn(
    "is_truly_high_freq",
    F.when(F.col("y_true") >= p80_threshold, 1).otherwise(0)
)

total_customers = df_preds.count()
total_truly_high = df_preds.filter(F.col("is_truly_high_freq") == 1).count()
base_rate = total_truly_high / total_customers

print(f"High-frequency threshold (p80): {p80_threshold:.1f} purchases/year")
print(f"Baseline rate: {base_rate:.1%}\n")

model_pred_cols = {
    "LR_drivers":  "y_pred_lr_drivers",
    "GBT_drivers": "y_pred_gbt_drivers",
    "LR_enhanced": "y_pred_lr_enhanced",
    "GBT_enhanced": "y_pred_gbt_enhanced"
}

k_percentiles = [0.05, 0.10, 0.20]

results = []

for model_name, pred_col in model_pred_cols.items():
    for k_pct in k_percentiles:
        k = int(total_customers * k_pct)
        threshold = df_preds.approxQuantile(pred_col, [1 - k_pct], 0.01)[0]
        
        top_k_df = df_preds.filter(F.col(pred_col) >= threshold)
        top_k_count = top_k_df.count()
        top_k_correct = top_k_df.filter(F.col("is_truly_high_freq") == 1).count()
        
        precision = top_k_correct / top_k_count if top_k_count > 0 else 0
        lift = precision / base_rate
        
        results.append({
            "model": model_name,
            "k_pct": k_pct,
            "precision": precision,
            "lift": lift
        })

print("Precision@K Results:\n")
for k_pct in k_percentiles:
    print(f"Top {k_pct:.0%}:")
    k_results = sorted([r for r in results if r["k_pct"] == k_pct], 
                       key=lambda x: x["precision"], reverse=True)
    for r in k_results:
        print(f"  {r['model']:<20} Precision: {r['precision']:>6.1%}  Lift: {r['lift']:>5.2f}x")
    print(f"  {'[Random baseline]':<20} Precision: {base_rate:>6.1%}  Lift: 1.00x\n")

PRECISION@K EVALUATION - Marketing Targeting Quality

📊 High-frequency buyer threshold (p80): 5.0 purchases/year
   Customers above this are considered 'truly high-frequency'

   Total customers: 1,305,446
   Truly high-frequency (top 20%): 311,859 (23.9%)
   Random targeting baseline: 23.9% precision

PRECISION@K RESULTS

🎯 TOP 5% (65,272 customers targeted):
   Model                Precision    Lift vs Random     True HV Found
   -----------------------------------------------------------------
   ⭐GBT_enhanced         86.6%       3.63x             67,254
     GBT_drivers          86.3%       3.61x             67,300
     LR_enhanced          86.2%       3.61x             66,910
     LR_drivers           85.9%       3.60x             67,201
   [Random baseline]      23.9%      1.00x              15,592

🎯 TOP 10% (130,544 customers targeted):
   Model                Precision    Lift vs Random     True HV Found
   -----------------------------------------------------------------
   ⭐

## Conclusion

### Best-Performing Model
The **Gradient Boosted Trees (Enhanced)** model achieved the best overall performance with the lowest RMSE and highest ranking accuracy. Adding historical baseline controls (past frequency, recency, and tenure) to the behavioral engagement drivers consistently improved prediction quality across both linear and tree-based approaches.

### Key Findings
1. **Enhanced feature set significantly outperforms drivers-only approach** – Adding log-transformed baseline controls reduced RMSE by 8-10% and improved ranking correlation
2. **GBT captures non-linear relationships better than linear regression** – Tree-based models demonstrated superior performance in both feature configurations
3. **Historical behavior remains the strongest predictor** – Past purchase frequency accounts for the majority of predictive power, but behavioral engagement drivers add meaningful incremental value
4. **Precision@K metrics validate targeting effectiveness** – The enhanced GBT model identifies high-frequency buyers 2-3x better than random selection when targeting the top 10% of customers

### Business Impact for Decathlon
Accurate purchase frequency prediction enables:
* **Optimized marketing spend** – Target high-potential customers with precision, reducing waste on low-propensity segments
* **Personalized customer experiences** – Tailor retention strategies, product recommendations, and communication frequency based on predicted engagement
* **Strategic resource allocation** – Focus CRM efforts on customers most likely to generate value, maximizing ROI
* **Lifetime value estimation** – Improved frequency forecasts enhance CLV calculations, supporting better business planning and customer segmentation

This model provides Decathlon with a data-driven foundation for customer lifetime value optimization and strategic marketing decisions.